In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *
from biked_commons.resource_utils import split_datasets_path
from biked_commons.conditioning import conditioning
from biked_commons.design_evaluation.scoring import *

c:\Users\Lyle\mambaforge\envs\pytorch_clip\lib\site-packages\sklearn\base.py:329: UserWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.1.3. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [2]:
data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)

#sample 100
data = data.sample(100, random_state=0)
data_tens = torch.tensor(data.values, dtype=torch.float32)

In [3]:
evaluator, requirement_names, requirement_types = construct_tensor_evaluator(StandardEvaluations, data.columns)
isobjective = torch.tensor(requirement_types) == 1


In [4]:
num_data = data.shape[0]
rider_condition = conditioning.sample_riders(num_data, split="test")
use_case_condition = conditioning.sample_use_case(num_data, split="test")
text_condition = conditioning.sample_text(num_data, split="test")
image_embeddings = conditioning.sample_image_embedding(num_data, split="test")
condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Embedding": image_embeddings}
# condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Text": text_condition}

In [12]:
eval_scores = evaluator(data_tens, condition)

c:\Users\Lyle\mambaforge\envs\pytorch_clip\lib\site-packages\sklearn\base.py:450: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [13]:
#check gradient of eval scores wrt data_tens
data_tens.requires_grad = True
eval_scores = evaluator(data_tens, condition)
eval_scores_sum = eval_scores.sum()
eval_scores_sum.backward()
print(data_tens.grad.shape)
print(data_tens.grad[0])



torch.Size([100, 97])
tensor([-5.2809e+00,  4.2750e+00,  3.5658e+00,  1.6447e+01, -7.6343e+00,
         1.1627e+00, -1.1589e+00, -1.1118e+01, -5.0858e+00, -2.0299e+00,
        -4.7824e-02,  7.1151e-03,  1.7680e-01,  5.3155e-02,  3.9809e-02,
         8.9996e-02, -3.1371e-04,  1.9956e+00, -6.2651e-03,  7.6468e+00,
         1.0421e-03,  1.6291e-02,  3.6730e-02, -3.3553e-01,  8.9014e-02,
        -4.9387e-03,  1.6675e-02,  3.3107e-02,  4.4194e-02,  1.3071e-01,
         4.1224e-01,  2.7335e-01, -1.5727e-02,  0.0000e+00,  1.0000e+00,
         0.0000e+00,  2.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
        -4.8677e-03, -3.5939e-03,  0.0000e+00, -6.6371e-03,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00, -2.8133e-01,  0.0000e+00, -2.0000e+00, -2.1668e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  9.4493e-01, -7.9803e-01,
         0.0000e+00,  0.0000e

c:\Users\Lyle\mambaforge\envs\pytorch_clip\lib\site-packages\sklearn\base.py:450: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [16]:
isobjective = torch.tensor(requirement_types) == 1
objective_scores = eval_scores[:, isobjective].detach().numpy()
# constraint_scores = eval_scores[:, ~isobjective].detach().numpy()

In [17]:
main_scorer = construct_scorer(MainScores, StandardEvaluations, data.columns)
detailed_scorer = construct_scorer(DetailedScores, StandardEvaluations, data.columns)

In [19]:
main_scorer(data_tens.detach(), condition)

c:\Users\Lyle\mambaforge\envs\pytorch_clip\lib\site-packages\sklearn\base.py:450: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Hypervolume                     0.000000
Constraint Satisfaction Rate    0.856154
Maximum Mean Discrepancy        0.003185
dtype: float64

In [20]:
detailed_scorer(data_tens.detach(), condition)

c:\Users\Lyle\mambaforge\envs\pytorch_clip\lib\site-packages\sklearn\base.py:450: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Min Objective Score: Usability Score - 0 to 1                                                                 0.791411
Min Objective Score: Drag Force                                                                              27.933560
Min Objective Score: Knee Angle Error                                                                       186.338130
Min Objective Score: Hip Angle Error                                                                        822.850400
Min Objective Score: Arm Angle Error                                                                        861.650940
Min Objective Score: Mass                                                                                    22.190498
Min Objective Score: Planar Compliance                                                                      180.347920
Min Objective Score: Transverse Compliance                                                                  265.026400
Min Objective Score: Eccentric Compliance       